# Preparation occurrence of ecosystems (res. 0.25°)

In [ ]:
# Libraries
import os
import xarray as xr

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

### Regridding potential occurance data to 0.025°

In [ ]:
# List of ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

In [ ]:
# Get land mask
ds_land = xr.open_zarr(f'{dir01}ds_prep_copernicus_land_mask_r001.zarr')

# Get potential occurance data
ds_pot_occur = xr.open_zarr(f'{dir02}ds_pot_occur_r001.zarr')

for etype in list_etype:
    # Replace nan-value with nan and change to dataset
    da_pot_occur = ds_pot_occur[etype].where(ds_land.copernicus_land_mask == 1)
    # Attributes of latitude and longitude
    da_pot_occur.lat.attrs = dict(standard_name = 'latitude',
                                  long_name = 'latitude',
                                  units = 'degrees_north',
                                  axis = 'Y')
    da_pot_occur.lon.attrs = dict(standard_name = 'longitude',
                                  long_name = 'longitude',
                                  units = 'degrees_east',
                                  axis = 'X')
    # Export
    da_pot_occur \
        .to_netcdf(f'{dir02}ds_pot_occur_{etype}_r001.nc', mode='w')

In [ ]:
%%bash 
cd ../paper_ecosystem_carbon/output/02_database_prep/

module load cdo

for i in trop_ef trop_df temp_ebf temp_dbf temp_enf temp_dnf glob_shrub \
    glob_grass glob_tundr glob_mount glob_crop glob_past
do
    cdo -P 24 remapycon,../data/luh2_v2h/states.nc \
    ds_pot_occur_${i}_r001.nc ds_pot_occur_${i}_r025.nc
done

In [ ]:
# Merge files
ds_pot_occur = xr.open_mfdataset(f'{dir02}ds_pot_occur_*_r025.nc')
# Set all grid cells with occurence to True and export
xr.where(ds_pot_occur > 0, True, False) \
    .to_zarr(f'{dir02}ds_pot_occur_r025.zarr', mode='w');

In [ ]:
# Delete intermediate files
for etype in list_etype:
    os.remove(f'{dir02}ds_pot_occur_{etype}_r001.nc')
    os.remove(f'{dir02}ds_pot_occur_{etype}_r025.nc')